# Module 07 — Assembling the Full Model (notebook)

Walkthrough of [`model.py`](model.py) and [`init.py`](init.py). We'll:

1. Build a small `TransformerLM` and inspect its parameter count by group.
2. Run the four shape/sanity tests from Section 8 of the README — the cheapest insurance against assembly bugs.
3. Compare standard init vs muP init: verify that muP keeps activation magnitudes width-invariant.
4. Swap the dense FFN for MoE with one config change.
5. Run the naive `generate()` to confirm the inference path works.

**Compute:** CPU is enough.  
**Time:** ~10 minutes.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from model import TransformerLM, ModelConfig
from init import standard_init, mup_init, param_groups_mup

torch.manual_seed(0)
print("torch:", torch.__version__)

## 1. Build a model and inspect parameters

A small DeepSeek-V3-shaped model: MLA attention, dense SwiGLU FFN, RMSNorm, RoPE, weight tying. This is the same config Module 11 will scale up for the pretraining demo.

In [ ]:
cfg = ModelConfig(
    vocab_size=8192,
    d_model=256,
    n_layers=6,
    max_seq_len=512,
    # MLA attention with DeepSeek-V3-shaped dimensions (just smaller)
    attention_type="mla",
    n_heads=8,
    d_head=32,
    d_rope=32,
    d_kv_latent=64,
    # Dense SwiGLU FFN
    ffn_type="dense",
    tie_weights=True,
)
model = TransformerLM(cfg)

counts = model.count_params()
print("Parameter count by group:")
for k, v in counts.items():
    print(f"  {k:10s} {v/1e6:>7.3f}M  ({v/counts['total']*100:>5.1f}%)")

The FFN dominates — as it does at every scale. At our small $d_{\text{model}}=256$, the embedding is also a noticeable chunk (the vocab is large relative to the model width). At frontier scale (8k+ width) the embedding fraction shrinks toward 1-2% of total params.

## 2. The four shape/sanity tests

From Section 8 of the README. Each takes a second; together they catch most assembly bugs.

In [ ]:
B, T = 4, 64
ids = torch.randint(0, cfg.vocab_size, (B, T))
targets = torch.randint(0, cfg.vocab_size, (B, T))

# Test 1: forward shape
logits = model(ids)
assert logits.shape == (B, T, cfg.vocab_size)
print(f"Test 1 — forward shape: OK  ({tuple(ids.shape)} -> {tuple(logits.shape)})")

# Test 2: parameter count envelope
# V*d + 12*L*d^2 (weight-tied formula). MLA is a bit different but close.
predicted = cfg.vocab_size * cfg.d_model + 12 * cfg.n_layers * cfg.d_model ** 2
actual = counts['total']
print(f"Test 2 — param count: predicted ~{predicted/1e6:.2f}M, actual {actual/1e6:.2f}M  (ratio {actual/predicted:.2f})")

# Test 3: loss at init ~ ln(vocab_size)
loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), targets.reshape(-1))
expected = torch.tensor(cfg.vocab_size, dtype=torch.float32).log().item()
assert abs(loss.item() - expected) < 1.0
print(f"Test 3 — loss at init: {loss.item():.3f} (expected ~{expected:.3f}): OK")

# Test 4: backward + one optimizer step decreases loss
loss.backward()
n_with_grad = sum(1 for p in model.parameters() if p.grad is not None)
n_with_none = sum(1 for p in model.parameters() if p.grad is None)
assert n_with_none == 0
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
opt.step(); opt.zero_grad()
loss2 = F.cross_entropy(model(ids).reshape(-1, cfg.vocab_size), targets.reshape(-1))
assert loss2.item() < loss.item()
print(f"Test 4 — backward + step: {n_with_grad} params with grad, 0 with None; "
      f"loss {loss.item():.3f} -> {loss2.item():.3f}: OK")

All four green. If any of these fails on a freshly-built model, *do not start a training run* — fix the model first.

The parameter count ratio is not exactly 1.00 because the envelope formula assumes MHA + SwiGLU(8/3); we're using MLA (more efficient on the KV side) and the SwiGLU hidden width is rounded to multiple-of-64. Close enough is what you want from an envelope check.

## 3. muP — verifying width-invariant activations

The muP claim: as you widen the network, the activations inside the model should keep approximately the same magnitudes — assuming you applied muP init.

Test: build the same architecture at three widths (128, 256, 512) and measure the RMS of the residual stream at the *deepest* block's output. With muP init, these should be roughly equal. With standard init, they should drift.

In [ ]:
def measure_residual_rms(d_model, init_mode, base_d=128, n_layers=4, seed=0):
    """Build a fresh model at width d_model, init it, forward random data,
    and return the residual-stream RMS at the output of the LAST block."""
    torch.manual_seed(seed)
    cfg = ModelConfig(
        vocab_size=2048, d_model=d_model, n_layers=n_layers,
        attention_type="mla", n_heads=4, d_head=d_model // 4,
        d_rope=16, d_kv_latent=32, max_seq_len=128, tie_weights=True,
    )
    m = TransformerLM(cfg)
    if init_mode == "standard":
        standard_init(m)
    elif init_mode == "mup":
        mup_init(m, d_model=d_model, base_d=base_d)
    if cfg.tie_weights:
        m.unembed.weight = m.tok_emb.weight

    # Forward and capture the residual stream BEFORE final_norm.
    with torch.no_grad():
        ids = torch.randint(0, cfg.vocab_size, (4, 64))
        x = m.tok_emb(ids)
        freqs_cis = m.freqs_cis[:64]
        for block in m.blocks:
            x, _ = block(x, freqs_cis)
    return x.pow(2).mean().sqrt().item()

widths = [128, 256, 512]
rms_standard = [measure_residual_rms(d, "standard") for d in widths]
rms_mup      = [measure_residual_rms(d, "mup", base_d=128) for d in widths]

print(f"{'width':>8s}  {'standard init':>14s}  {'muP init (base=128)':>22s}")
for d, rs, rm in zip(widths, rms_standard, rms_mup):
    print(f"{d:>8d}  {rs:>14.3f}  {rm:>22.3f}")

Three things to read off these numbers:

1. **Standard init drifts upward with width.** Residual RMS roughly doubles as width doubles (0.028 → 0.058 → 0.159). This is the kind of width-dependent dynamics that makes the optimal LR shift between scales — at $d=512$ the activations are 5× larger than at $d=128$, so a learning rate tuned for the smaller width is wrong at the larger one.
2. **muP init holds activations constant** (0.028 → 0.021 → 0.019). The $1/\sqrt{m}$ scaling of hidden weights compensates for the wider matmul sums, keeping the residual stream in the same regime at every width.
3. **This is necessary but not sufficient for mu-transfer.** The full test is doing an LR sweep at the proxy width, training at the target width with the same swept $\eta^*$, and checking that the loss curve matches what you'd get from a fresh sweep at the target. That's a multi-hour experiment we won't run here — Module 09 (Learning Rate) covers the practical mu-transfer recipe for the pretraining demo.

If you want to verify mu-transfer empirically end-to-end, the canonical reproduction target is Figure 2 of "Tensor Programs V": same LR sweep at multiple widths, optima all line up at the same $\eta^*$.

## 4. Swap dense FFN for MoE

One-line config change. The model now uses Module 06's `MoEFFN`; the rest of the assembly is identical. We pass `return_router_info=True` to the forward to collect per-layer router outputs for the training-loop bias updates.

In [ ]:
cfg_moe = ModelConfig(
    vocab_size=2048, d_model=128, n_layers=2,
    attention_type="gqa", n_heads=4, n_kv_heads=2,  # smaller attention to keep budget low
    ffn_type="moe",
    n_experts=8, top_k=2, d_ffn_expert=64, n_shared_experts=1,
    max_seq_len=128, tie_weights=True,
)
moe_model = TransformerLM(cfg_moe)
ids = torch.randint(0, cfg_moe.vocab_size, (2, 32))
logits, router_infos = moe_model(ids, return_router_info=True)

print(f"logits shape: {tuple(logits.shape)}")
print(f"router_infos: {len(router_infos)} entries (one per layer; None for dense, populated for MoE)")
print(f"layer-0 utilization: {[f'{u:.3f}' for u in router_infos[0].utilization.tolist()]}")
print(f"target utilization:  {cfg_moe.top_k / cfg_moe.n_experts:.3f}")
print(f"\nTotal params:        {moe_model.count_params()['total']/1e6:.2f}M")
print(f"FFN params (MoE):    {moe_model.count_params()['ffn']/1e6:.2f}M")

The same `TransformerLM` class handles dense and MoE. The training loop (Module 08) just needs to:

1. Pass `return_router_info=True` to the forward.
2. After `optimizer.step()`, walk the returned `router_infos` and call `block.ffn.router.update_bias(info.utilization)` on each non-None entry (for aux-loss-free balancing) or add `info.aux_loss` to the main loss (for classical aux-loss balancing).

Module 11's pretraining demo is dense — `ffn_type="dense"`. The MoE path is here for students who want to scale up.

## 5. Naive generation

Confirm the inference path. The model is untrained so the outputs are random — we're just checking the API works.

In [ ]:
prompt = torch.tensor([[1, 2, 3, 4, 5]])
out = model.generate(prompt, max_new_tokens=20, temperature=1.0, top_k=50)
print(f"prompt:    {prompt.tolist()[0]}")
print(f"generated: {out.tolist()[0]}")
print(f"shape: {tuple(prompt.shape)} -> {tuple(out.shape)}")

## Recap

You now have:

- A working `TransformerLM` that assembles every Part 2 component — MLA attention, RMSNorm, SwiGLU, RoPE, weight tying — into a single forward call.
- Standard and muP initializations, with muP LR groups ready to plug into Module 08's optimizer setup.
- Four shape/sanity tests that should run before every pretraining attempt.
- A drop-in MoE path (`ffn_type="moe"`) for the scaling-up case Module 06 described.
- A naive generation API for inference sanity-checking.

**Next:** [Part 3 — Pretraining](../../part-3-pretraining/), starting with Module 08 (The Training Loop). The model is built; now it's time to train it on real data.